# 얼굴 + 차량번호판 모자이크 — 오픈소스 모델 4종 비교

영상 경로 **하나**를 지정하면 아래 4가지 방식으로 각각 돌려서 결과 영상을 따로 저장합니다.

| # | 방식 | 탐지기 | 라이선스 | 비고 |
|---|---|---|---|---|
| A | **Ultralytics YOLO** (직접 구현) | face YOLO + plate YOLO | AGPL-3.0 | 가장 빠름, 튜닝 자유도 최고 |
| B | **Meta EgoBlur Gen2** | Faster R-CNN (TorchScript) | Apache 2.0 | 정확도 최상, 무거움 |
| C | **dashcam_anonymizer** | YOLOv8 커스텀 단일 모델 | 오픈소스 | 얼굴+번호판 한 모델 |
| D | **video-privacy-blur** | YOLO face + YOLO plate (CLI) | 오픈소스 | CLI 한 줄, 픽셀레이션 지원 |

> 환경 가정: Windows / Python 3.11 / RTX 5060 Laptop (CUDA 12.8) / conda 또는 venv


---
## 0. 설정 — 여기만 수정하면 됩니다

In [1]:
from pathlib import Path
import os, sys, subprocess, shutil, time, json

# ===== 여기만 수정 =====
INPUT_VIDEO = Path(r"C:\Users\Win11Pro\Desktop\새 폴더\KakaoTalk_20260723_094854803.mp4")   # 원본 영상 경로
OUTPUT_DIR  = Path(r"C:\Users\Win11Pro\Desktop\새 폴더\결과")     # 결과 저장 폴더
WORK_DIR    = Path(r"C:\Users\Win11Pro\Desktop\새 폴더\결과1")    # 레포/가중치 다운로드 폴더
DEVICE      = "0"                            # GPU 0 / CPU면 "cpu"
MOSAIC_MODE = "pixelate"                     # "pixelate" | "gaussian"
BLOCKS      = 12                             # 모자이크 블록 수 (작을수록 굵게 뭉갬)
BOX_SCALE   = 1.20                           # 박스 확대 배율 (경계 누출 방지)
CONF_FACE   = 0.10                           # 재현율 우선 → 낮게
CONF_PLATE  = 0.10
IMGSZ       = 1280                           # 작은 객체 대응
MAX_FRAMES  = None                           # 테스트용 프레임 제한 (예: 300), 전체는 None
# ======================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = WORK_DIR / "models"; MODELS_DIR.mkdir(exist_ok=True)

assert INPUT_VIDEO.exists(), f"입력 영상이 없습니다: {INPUT_VIDEO}"
print("입력 :", INPUT_VIDEO)
print("출력 :", OUTPUT_DIR)
print("작업 :", WORK_DIR)

입력 : C:\Users\Win11Pro\Desktop\새 폴더\KakaoTalk_20260723_094854803.mp4
출력 : C:\Users\Win11Pro\Desktop\새 폴더\결과
작업 : C:\Users\Win11Pro\Desktop\새 폴더\결과1


In [2]:
# 공통 유틸: 다운로드 / 셸 실행 / 영상 메타
import urllib.request

def download(url: str, dst: Path, desc: str = "") -> Path:
    """이미 있으면 스킵. 실패 시 예외를 그대로 올려서 원인 파악이 쉽게."""
    if dst.exists() and dst.stat().st_size > 0:
        print(f"[skip] 이미 존재: {dst.name}")
        return dst
    print(f"[get ] {desc or dst.name} <- {url}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, dst)
    except Exception as e:
        raise RuntimeError(f"다운로드 실패({dst.name}). 브라우저로 직접 받아 {dst}에 두세요.\n  {e}")
    print(f"[ok  ] {dst.name}  ({dst.stat().st_size/1e6:.1f} MB)")
    return dst

def run(cmd, cwd=None):
    """서브프로세스 실행 + 실시간 출력. 실패해도 노트북이 죽지 않게 returncode 반환."""
    print(">", " ".join(str(c) for c in cmd))
    p = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-3000:])
    if p.returncode != 0:
        print("[FAIL]", p.stderr[-3000:])
    return p.returncode

def video_info(path: Path):
    import cv2
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"영상을 열 수 없습니다: {path}")
    info = dict(fps=cap.get(cv2.CAP_PROP_FPS) or 30,
                w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.release()
    return info

RESULTS = {}   # {방식명: {"path":..., "sec":..., "ok":bool, "note":...}}

def record(name, path, t0, ok=True, note=""):
    RESULTS[name] = dict(path=str(path), sec=round(time.time()-t0, 1), ok=ok, note=note)
    print(f"[{name}] {'OK' if ok else 'FAIL'}  {RESULTS[name]['sec']}s  -> {path}")

In [3]:
# 의존성 설치 (한 번만)
%pip install -q ultralytics opencv-python huggingface_hub
import cv2, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("opencv", cv2.__version__)
print(video_info(INPUT_VIDEO))


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
torch 2.12.0.dev20260408+cu128 | cuda: True | NVIDIA GeForce RTX 5060 Laptop GPU
opencv 4.10.0
{'fps': 24.0, 'w': 1332, 'h': 720, 'n': 389}


---
## A. Ultralytics YOLO 직접 구현 (face + plate)

가장 빠르고 파라미터를 마음대로 만질 수 있는 방식입니다.
얼굴은 WIDERFace로 학습된 face 전용 가중치, 번호판은 공개 plate 가중치를 씁니다.

**모델 버전 관련**: 여기서 쓰는 `yolov12*-face.pt` / `license_plate_detector.pt`는 커뮤니티가
파인튜닝해서 배포한 **가중치 파일**이라 아키텍처가 v8/v11/v12로 고정입니다.
YOLO26으로 돌리고 싶다면 직접 파인튜닝해야 하며, 그 방법은 마지막 섹션에 적어뒀습니다.

In [4]:
# A-1. 가중치 준비
FACE_W  = MODELS_DIR / "yolov12m-face.pt"
PLATE_W = MODELS_DIR / "license_plate_detector.pt"

# akanametov/yolo-face 릴리스 (WIDERFace 학습). 링크가 바뀌면 레포 Releases 탭에서 직접 받으세요.
FACE_URL  = "https://github.com/akanametov/yolo-face/releases/download/v0.0.0/yolov12m-face.pt"
# license_plate_detector.pt : Muhammad-Zeerak-Khan/Automatic-License-Plate-Recognition-using-YOLOv8 배포본
PLATE_URL = "https://github.com/MengWoods/video-privacy-blur/releases/download/v0.1.0/license_plate_detector.pt"

for url, dst in [(FACE_URL, FACE_W), (PLATE_URL, PLATE_W)]:
    try:
        download(url, dst)
    except Exception as e:
        print(f"[WARN] {e}\n  -> 수동 다운로드 후 {dst} 위치에 놓고 이 셀을 다시 실행하세요.")

[get ] yolov12m-face.pt <- https://github.com/akanametov/yolo-face/releases/download/v0.0.0/yolov12m-face.pt
[WARN] 다운로드 실패(yolov12m-face.pt). 브라우저로 직접 받아 C:\Users\Win11Pro\Desktop\새 폴더\결과1\models\yolov12m-face.pt에 두세요.
  HTTP Error 404: Not Found
  -> 수동 다운로드 후 C:\Users\Win11Pro\Desktop\새 폴더\결과1\models\yolov12m-face.pt 위치에 놓고 이 셀을 다시 실행하세요.
[get ] license_plate_detector.pt <- https://github.com/MengWoods/video-privacy-blur/releases/download/v0.1.0/license_plate_detector.pt
[WARN] 다운로드 실패(license_plate_detector.pt). 브라우저로 직접 받아 C:\Users\Win11Pro\Desktop\새 폴더\결과1\models\license_plate_detector.pt에 두세요.
  HTTP Error 404: Not Found
  -> 수동 다운로드 후 C:\Users\Win11Pro\Desktop\새 폴더\결과1\models\license_plate_detector.pt 위치에 놓고 이 셀을 다시 실행하세요.


In [5]:
# A-2. 모자이크 유틸
import numpy as np

def pixelate(img, x1, y1, x2, y2, blocks=BLOCKS):
    h, w = img.shape[:2]
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(w, int(x2)), min(h, int(y2))
    if x2 <= x1 or y2 <= y1:
        return img
    roi = img[y1:y2, x1:x2]
    small = cv2.resize(roi,
                       (max(1, (x2 - x1)//blocks), max(1, (y2 - y1)//blocks)),
                       interpolation=cv2.INTER_LINEAR)
    img[y1:y2, x1:x2] = cv2.resize(small, (x2-x1, y2-y1), interpolation=cv2.INTER_NEAREST)
    return img

def gaussian(img, x1, y1, x2, y2):
    h, w = img.shape[:2]
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(w, int(x2)), min(h, int(y2))
    if x2 <= x1 or y2 <= y1:
        return img
    roi = img[y1:y2, x1:x2]
    k = max(3, (min(x2-x1, y2-y1)//4)|1)          # 홀수 커널
    img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)
    return img

APPLY = pixelate if MOSAIC_MODE == "pixelate" else gaussian

def expand_box(b, scale=BOX_SCALE):
    x1, y1, x2, y2 = b
    cx, cy = (x1+x2)/2, (y1+y2)/2
    bw, bh = (x2-x1)*scale, (y2-y1)*scale
    return cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2

In [6]:
# A-3. 실행
from ultralytics import YOLO

OUT_A = OUTPUT_DIR / "A_ultralytics.mp4"
t0 = time.time()
try:
    face_model  = YOLO(str(FACE_W))
    plate_model = YOLO(str(PLATE_W))

    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    meta = video_info(INPUT_VIDEO)
    writer = cv2.VideoWriter(str(OUT_A), cv2.VideoWriter_fourcc(*"mp4v"),
                             meta["fps"], (meta["w"], meta["h"]))
    n, hits = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok or (MAX_FRAMES and n >= MAX_FRAMES):
            break
        boxes = []
        for model, conf in [(face_model, CONF_FACE), (plate_model, CONF_PLATE)]:
            r = model.predict(frame, conf=conf, imgsz=IMGSZ,
                              device=DEVICE, verbose=False)[0]
            boxes += [b.tolist() for b in r.boxes.xyxy.cpu()]
        for b in boxes:
            frame = APPLY(frame, *expand_box(b))
        hits += len(boxes)
        writer.write(frame); n += 1
        if n % 100 == 0:
            print(f"  {n} frames / 누적 검출 {hits}")
    cap.release(); writer.release()
    record("A_ultralytics", OUT_A, t0, ok=True, note=f"{n}f, 검출 {hits}")
except Exception as e:
    record("A_ultralytics", OUT_A, t0, ok=False, note=str(e))
    print("[ERROR]", e)

[A_ultralytics] FAIL  1.0s  -> C:\Users\Win11Pro\Desktop\새 폴더\결과\A_ultralytics.mp4
[ERROR] [Errno 2] No such file or directory: 'C:\\Users\\Win11Pro\\Desktop\\새 폴더\\결과1\\models\\yolov12m-face.pt'


---
## B. Meta EgoBlur Gen2 (Apache 2.0)

얼굴/번호판 각각 전용 Faster R-CNN 모델. 상업적 이용까지 자유롭고 정확도가 가장 좋습니다.

⚠️ **모델 가중치는 자동 다운로드가 불가능합니다.**
[projectaria.com/tools/egoblur](https://www.projectaria.com/tools/egoblur/) 에서 이메일을 넣고
라이선스에 동의하면 다운로드 링크가 나옵니다. 받은 `.zip`을 풀어서 아래 경로에 두세요.

```
{MODELS_DIR}/egoblur/ego_blur_face.jit
{MODELS_DIR}/egoblur/ego_blur_lp.jit
```

In [7]:
# B-1. 레포 클론 + 설치
EGO_DIR   = WORK_DIR / "EgoBlur"
EGO_MODEL = MODELS_DIR / "egoblur"
EGO_MODEL.mkdir(exist_ok=True)

if not EGO_DIR.exists():
    run(["git", "clone", "https://github.com/facebookresearch/EgoBlur.git", str(EGO_DIR)])
else:
    print("[skip] 이미 클론됨")

# CLI 설치 (torch/torchvision은 기존 CUDA 빌드를 그대로 씁니다)
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EGO_DIR)])

face_jit = EGO_MODEL / "ego_blur_face.jit"
lp_jit   = EGO_MODEL / "ego_blur_lp.jit"
for f in (face_jit, lp_jit):
    print(("[ok  ] " if f.exists() else "[없음] "), f)

> git clone https://github.com/facebookresearch/EgoBlur.git C:\Users\Win11Pro\Desktop\새 폴더\결과1\EgoBlur


Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Win11Pro\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Win11Pro\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Win11Pro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 40: illegal multibyte sequence


> C:\Project\PythonProject\PyTorch001\.venv\Scripts\python.exe -m pip install -q -e C:\Users\Win11Pro\Desktop\새 폴더\결과1\EgoBlur
[FAIL] <module>
      main()
    File "C:\Users\Win11Pro\AppData\Local\Temp\pip-install-h9odd6y6\vrs_d84879c9b14a4c9f8283409073a16e86\setup.py", line 122, in main
      setup(
    File "C:\Project\PythonProject\PyTorch001\.venv\Lib\site-packages\setuptools\__init__.py", line 87, in setup
      return distutils.core.setup(**attrs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "C:\Project\PythonProject\PyTorch001\.venv\Lib\site-packages\setuptools\_distutils\core.py", line 185, in setup
      return run_commands(dist)
             ^^^^^^^^^^^^^^^^^^
    File "C:\Project\PythonProject\PyTorch001\.venv\Lib\site-packages\setuptools\_distutils\core.py", line 201, in run_commands
      dist.run_commands()
    File "C:\Project\PythonProject\PyTorch001\.venv\Lib\site-packages\setuptools\_distutils\dist.py", line 968, in run_commands
      self.run_command(cmd)
   

In [8]:
# B-2. 실행
OUT_B = OUTPUT_DIR / "B_egoblur.mp4"
t0 = time.time()

if not (face_jit.exists() and lp_jit.exists()):
    record("B_egoblur", OUT_B, t0, ok=False,
           note="모델 미다운로드 — projectaria.com/tools/egoblur 에서 수동 다운로드 필요")
else:
    # gen2 CLI. 버전에 따라 진입점이 다를 수 있어 두 가지를 순차 시도합니다.
    args = ["--face_model_path", str(face_jit),
            "--lp_model_path",   str(lp_jit),
            "--input_video_path",  str(INPUT_VIDEO),
            "--output_video_path", str(OUT_B),
            "--face_model_score_threshold", "0.10",
            "--lp_model_score_threshold",   "0.10",
            "--scale_factor_detections", str(BOX_SCALE)]
    rc = run(["egoblur-gen2", *args])
    if rc != 0:
        rc = run([sys.executable, str(EGO_DIR/"gen2"/"tools"/"demo_egoblur.py"), *args])
    record("B_egoblur", OUT_B, t0, ok=(rc == 0 and OUT_B.exists()),
           note="CLI 인자명은 레포 README 기준 (버전업 시 확인)")

[B_egoblur] FAIL  0.0s  -> C:\Users\Win11Pro\Desktop\새 폴더\결과\B_egoblur.mp4


---
## C. dashcam_anonymizer

얼굴 + 번호판을 **하나의 YOLOv8 커스텀 모델**로 잡습니다.
원본은 이미지 폴더 기반이라, 여기서는 영상 → 프레임 → 모자이크 → 영상 재조립으로 감쌌습니다.

In [10]:
# C-1. 레포 + 모델 준비
DC_DIR = WORK_DIR / "dashcam_anonymizer"
DC_W   = MODELS_DIR / "dashcam_yolov8.pt"

if not DC_DIR.exists():
    run(["git", "clone", "https://github.com/varungupta31/dashcam_anonymizer.git", str(DC_DIR)])

# setup.sh 가 gdown으로 받아오는 그 가중치입니다. 리눅스 셸이 아니면 아래처럼 gdown 직접 사용.
%pip install -q gdown
if not DC_W.exists():
    print("레포 README의 Google Drive 링크에서 모델을 받아 아래 경로에 두세요:")
    print("  ", DC_W)
    print("  (setup.sh 안의 gdown 명령을 그대로 써도 됩니다)")
    # 예시: !gdown <FILE_ID> -O "{DC_W}"
else:
    print("[ok] ", DC_W)

Note: you may need to restart the kernel to use updated packages.
레포 README의 Google Drive 링크에서 모델을 받아 아래 경로에 두세요:
   C:\Users\Win11Pro\Desktop\새 폴더\결과1\models\dashcam_yolov8.pt
  (setup.sh 안의 gdown 명령을 그대로 써도 됩니다)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# C-2. 영상 래퍼로 실행
OUT_C = OUTPUT_DIR / "C_dashcam.mp4"
t0 = time.time()
try:
    if not DC_W.exists():
        raise FileNotFoundError("dashcam_anonymizer 가중치가 없습니다 (위 셀 안내 참고)")

    dc_model = YOLO(str(DC_W))
    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    meta = video_info(INPUT_VIDEO)
    writer = cv2.VideoWriter(str(OUT_C), cv2.VideoWriter_fourcc(*"mp4v"),
                             meta["fps"], (meta["w"], meta["h"]))
    n, hits = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok or (MAX_FRAMES and n >= MAX_FRAMES):
            break
        r = dc_model.predict(frame, conf=0.10, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        for b in r.boxes.xyxy.cpu():
            frame = APPLY(frame, *expand_box(b.tolist()))
            hits += 1
        writer.write(frame); n += 1
    cap.release(); writer.release()
    record("C_dashcam", OUT_C, t0, ok=True, note=f"{n}f, 검출 {hits}")
except Exception as e:
    record("C_dashcam", OUT_C, t0, ok=False, note=str(e))
    print("[ERROR]", e)

[C_dashcam] FAIL  0.0s  -> C:\Users\Win11Pro\Desktop\새 폴더\결과\C_dashcam.mp4
[ERROR] dashcam_anonymizer 가중치가 없습니다 (위 셀 안내 참고)


---
## D. video-privacy-blur (CLI)

face YOLO + plate YOLO를 CLI 한 줄로 돌립니다. 픽셀레이션 강도를 인자로 조절할 수 있습니다.

In [ ]:
# D-1. 설치
VPB_DIR = WORK_DIR / "video-privacy-blur"
if not VPB_DIR.exists():
    run(["git", "clone", "https://github.com/MengWoods/video-privacy-blur.git", str(VPB_DIR)])
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(VPB_DIR/"requirements.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(VPB_DIR)])

# 이 툴은 자체 models/ 폴더를 봅니다. A에서 받은 가중치를 복사해 재사용.
VPB_MODELS = VPB_DIR / "models"; VPB_MODELS.mkdir(exist_ok=True)
VPB_FACE  = VPB_MODELS / "yolov8n-face.pt"
VPB_PLATE = VPB_MODELS / "license_plate_detector.pt"

try:
    download("https://github.com/akanametov/yolo-face/releases/download/v0.0.0/yolov8n-face.pt", VPB_FACE)
except Exception as e:
    print("[WARN]", e)
if PLATE_W.exists() and not VPB_PLATE.exists():
    shutil.copy(PLATE_W, VPB_PLATE); print("[ok] plate 가중치 복사")

In [ ]:
# D-2. 실행
OUT_D = OUTPUT_DIR / "D_privacyblur.mp4"
t0 = time.time()

cmd = ["privacy-blur",
       "--plate-weights", str(VPB_PLATE),
       "--face-detector", "yolo",
       "--face-yolo-weights", str(VPB_FACE),
       "--input", str(INPUT_VIDEO),
       "--output", str(OUT_D),
       "--device", "cuda" if DEVICE != "cpu" else "cpu",
       "--imgsz", str(IMGSZ),
       "--conf", str(CONF_FACE),
       "--scale", str(BOX_SCALE),
       "--pixelate-blocks", str(BLOCKS)]

rc = run(cmd)
if rc != 0:   # 콘솔 스크립트가 PATH에 없을 때 모듈로 재시도
    rc = run([sys.executable, "-m", "privacy_blur", *cmd[1:]])
record("D_privacyblur", OUT_D, t0, ok=(rc == 0 and OUT_D.exists()),
       note="인자명은 레포 README 기준")

---
## E. 결과 요약 & 육안 비교

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS).T[["ok", "sec", "path", "note"]]
display(df)
(OUTPUT_DIR / "summary.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2),
                                          encoding="utf-8")
print("\n요약 저장:", OUTPUT_DIR / "summary.json")

In [ ]:
# 같은 프레임을 방식별로 나란히 뽑아 비교 (검출 누락 확인용)
import matplotlib.pyplot as plt

FRAME_IDX = 100   # 확인하고 싶은 프레임 번호

def grab(path, idx):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = cap.read(); cap.release()
    return cv2.cvtColor(f, cv2.COLOR_BGR2RGB) if ok else None

panels = [("원본", INPUT_VIDEO)] + [(k, v["path"]) for k, v in RESULTS.items() if v["ok"]]
fig, axes = plt.subplots(1, len(panels), figsize=(6*len(panels), 5))
axes = np.atleast_1d(axes)
for ax, (name, p) in zip(axes, panels):
    img = grab(p, FRAME_IDX)
    ax.imshow(img) if img is not None else ax.text(.5, .5, "read fail", ha="center")
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

---
## F. YOLO26으로 직접 학습하고 싶다면

`ultralytics` 패키지는 YOLO26을 정식 지원하므로 파인튜닝만 하면 바로 위 A 섹션에 꽂아 쓸 수 있습니다.
공개 face/plate 가중치가 v8/v11/v12뿐이라 A에서 그걸 쓴 것뿐입니다.

```python
from ultralytics import YOLO
model = YOLO("yolo26m.pt")          # NMS-free, end-to-end
model.train(
    data="face_plate.yaml",          # names: [face, plate]
    epochs=100, imgsz=1280,          # 작은 객체 → 고해상도 필수
    batch=8, device=0,
    hsv_v=0.5, degrees=5, scale=0.6, # 야간/원거리 대응 증강
)
```

**데이터 조달**
- 얼굴: WIDERFace (공개, 소형 얼굴 다수)
- 번호판: UC3M-LP 등 공개 셋 + 국내 번호판은 합성 데이터로 증강
- 이미 가지고 계신 AI-Hub 차량 데이터에 번호판 bbox만 추가 라벨링하면 국내 도메인 보강에 효과가 큽니다

**모자이크 목적 튜닝 원칙**: precision보다 **recall**. conf를 0.05까지 내리고 박스를 1.2~1.3배 키우세요.
하나만 놓쳐도 비식별화는 실패입니다.
